In [1]:
import csv
import requests

API_KEY = "n9Z8sXttsue83foh2ihaLFO910uRxFMzZjRNGwXb"
URL = f"https://api.sportradar.com/tennis/trial/v3/en/competitions.json?api_key={API_KEY}"
HEADERS = {"accept": "application/json"}

try:
    response = requests.get(URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    data = response.json()

    competitions = data.get("competitions", [])

    # Dictionaries/Lists to avoid duplicate category entries
    categories_dict = {}
    competitions_list = []

    for comp in competitions:
        category = comp.get("category", {})
        cat_id = category.get("id", "")
        cat_name = category.get("name", "")

        # Collect unique categories
        if cat_id and cat_id not in categories_dict:
            categories_dict[cat_id] = cat_name

        # Collect competition records matching exact SQL column names
        competitions_list.append([
            comp.get("id", ""),
            comp.get("name", ""),
            comp.get("parent_id", None),  # Preserves NULL for parent_id if missing
            comp.get("type", ""),
            comp.get("gender", ""),
            cat_id  # Foreign key reference to Categories table
        ])

    # 1. Write categories.csv
    with open("categories.csv", "w", newline="", encoding="utf-8") as f_cat:
        writer = csv.writer(f_cat)
        writer.writerow(["category_id", "category_name"])
        for cat_id, cat_name in categories_dict.items():
            writer.writerow([cat_id, cat_name])

    # 2. Write competitions.csv
    with open("competitions.csv", "w", newline="", encoding="utf-8") as f_comp:
        writer = csv.writer(f_comp)
        writer.writerow(["competition_id", "competition_name", "parent_id", "type", "gender", "category_id"])
        writer.writerows(competitions_list)

    print(f"✅ Success:")
    print(f"   - Saved {len(categories_dict)} categories to categories.csv")
    print(f"   - Saved {len(competitions_list)} competitions to competitions.csv")

except requests.exceptions.RequestException as e:
    print(f"❌ Request failed: {e}")

✅ Success:
   - Saved 18 categories to categories.csv
   - Saved 6619 competitions to competitions.csv


In [2]:
import sqlite3
import pandas as pd

# Load CSVs directly into your relational SQL database
conn = sqlite3.connect("tennis_analytics.db")

df_categories = pd.read_csv("categories.csv")
df_competitions = pd.read_csv("competitions.csv")

df_categories.to_sql("Categories", conn, if_exists="append", index=False)
df_competitions.to_sql("Competitions", conn, if_exists="append", index=False)

conn.close()
print("Data imported into SQL database successfully!")

Data imported into SQL database successfully!


In [3]:
import csv
import requests

API_KEY = "n9Z8sXttsue83foh2ihaLFO910uRxFMzZjRNGwXb"
URL = f"https://api.sportradar.com/tennis/trial/v3/en/complexes.json?api_key={API_KEY}"
HEADERS = {"accept": "application/json"}

try:
    response = requests.get(URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    data = response.json()

    complexes_data = data.get("complexes", [])

    complexes_list = []
    venues_list = []

    for comp in complexes_data:
        complex_id = comp.get("id", "")
        complex_name = comp.get("name", "")

        # 1. Collect Complex Details
        complexes_list.append([complex_id, complex_name])

        # 2. Collect Linked Venues
        venues = comp.get("venues", [])
        for venue in venues:
            venues_list.append([
                venue.get("id", ""),
                venue.get("name", ""),
                venue.get("city_name", "N/A"),
                venue.get("country_name", "N/A"),
                venue.get("country_code", "N/A"),
                venue.get("timezone", "UTC"),
                complex_id  # Foreign Key linking to Complexes table
            ])

    # Save to complexes.csv
    with open("complexes.csv", "w", newline="", encoding="utf-8") as f_comp:
        writer = csv.writer(f_comp)
        writer.writerow(["complex_id", "complex_name"])
        writer.writerows(complexes_list)

    # Save to venues.csv
    with open("venues.csv", "w", newline="", encoding="utf-8") as f_venue:
        writer = csv.writer(f_venue)
        writer.writerow(["venue_id", "venue_name", "city_name", "country_name", "country_code", "timezone", "complex_id"])
        writer.writerows(venues_list)

    print(f"✅ Success:")
    print(f"   - Saved {len(complexes_list)} records to complexes.csv")
    print(f"   - Saved {len(venues_list)} records to venues.csv")

except requests.exceptions.RequestException as e:
    print(f"❌ Request failed: {e}")

✅ Success:
   - Saved 767 records to complexes.csv
   - Saved 4021 records to venues.csv


In [7]:
import csv
import requests

API_KEY = "n9Z8sXttsue83foh2ihaLFO910uRxFMzZjRNGwXb"

# Fixed endpoint path: 'double_competitors_rankings.json'
URL = f"https://api.sportradar.com/tennis/trial/v3/en/double_competitors_rankings.json?api_key={API_KEY}"
HEADERS = {"accept": "application/json"}

try:
    response = requests.get(URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    data = response.json()

    rankings_groups = data.get("rankings", [])

    competitors_dict = {}
    rankings_list = []

    for group in rankings_groups:
        competitor_rankings = group.get("competitor_rankings", [])
        
        for comp_rank in competitor_rankings:
            competitor = comp_rank.get("competitor", {})
            comp_id = competitor.get("id", "")
            
            # 1. Collect Unique Competitor Details
            if comp_id and comp_id not in competitors_dict:
                competitors_dict[comp_id] = [
                    comp_id,
                    competitor.get("name", "N/A"),
                    competitor.get("country", "N/A"),
                    competitor.get("country_code", "N/A"),
                    competitor.get("abbreviation", "N/A")
                ]

            # 2. Collect Ranking Entry Details
            rankings_list.append([
                comp_rank.get("rank", 0),
                comp_rank.get("movement", 0),
                comp_rank.get("points", 0),
                comp_rank.get("competitions_played", 0),
                comp_id  # Foreign Key linking to Competitors table
            ])

    # Save to competitors.csv
    with open("competitors.csv", "w", newline="", encoding="utf-8") as f_comp:
        writer = csv.writer(f_comp)
        writer.writerow(["competitor_id", "name", "country", "country_code", "abbreviation"])
        for comp_data in competitors_dict.values():
            writer.writerow(comp_data)

    # Save to competitor_rankings.csv
    with open("competitor_rankings.csv", "w", newline="", encoding="utf-8") as f_rank:
        writer = csv.writer(f_rank)
        writer.writerow(["rank", "movement", "points", "competitions_played", "competitor_id"])
        writer.writerows(rankings_list)

    print("✅ Success:")
    print(f"   - Saved {len(competitors_dict)} competitors to competitors.csv")
    print(f"   - Saved {len(rankings_list)} ranking entries to competitor_rankings.csv")

except requests.exceptions.RequestException as e:
    print(f"❌ Request failed: {e}")

✅ Success:
   - Saved 1000 competitors to competitors.csv
   - Saved 1000 ranking entries to competitor_rankings.csv


In [9]:
import sqlite3
import pandas as pd

DB_NAME = "tennis_analytics.db"

def build_database():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    # Enable foreign keys
    cursor.execute("PRAGMA foreign_keys = ON;")

    # 1. Create Tables with Schema Constraints
    cursor.executescript("""
    DROP TABLE IF EXISTS Competitor_Rankings;
    DROP TABLE IF EXISTS Competitors;
    DROP TABLE IF EXISTS Venues;
    DROP TABLE IF EXISTS Complexes;
    DROP TABLE IF EXISTS Competitions;
    DROP TABLE IF EXISTS Categories;

    CREATE TABLE Categories (
        category_id VARCHAR(50) PRIMARY KEY,
        category_name VARCHAR(100) NOT NULL
    );

    CREATE TABLE Competitions (
        competition_id VARCHAR(50) PRIMARY KEY,
        competition_name VARCHAR(100) NOT NULL,
        parent_id VARCHAR(50),
        type VARCHAR(20) NOT NULL,
        gender VARCHAR(10) NOT NULL,
        category_id VARCHAR(50),
        FOREIGN KEY (category_id) REFERENCES Categories(category_id)
    );

    CREATE TABLE Complexes (
        complex_id VARCHAR(50) PRIMARY KEY,
        complex_name VARCHAR(100) NOT NULL
    );

    CREATE TABLE Venues (
        venue_id VARCHAR(50) PRIMARY KEY,
        venue_name VARCHAR(100) NOT NULL,
        city_name VARCHAR(100) NOT NULL,
        country_name VARCHAR(100) NOT NULL,
        country_code CHAR(3) NOT NULL,
        timezone VARCHAR(100) NOT NULL,
        complex_id VARCHAR(50),
        FOREIGN KEY (complex_id) REFERENCES Complexes(complex_id)
    );

    CREATE TABLE Competitors (
        competitor_id VARCHAR(50) PRIMARY KEY,
        name VARCHAR(100) NOT NULL,
        country VARCHAR(100) NOT NULL,
        country_code CHAR(3) NOT NULL,
        abbreviation VARCHAR(10) NOT NULL
    );

    CREATE TABLE Competitor_Rankings (
        rank_id INTEGER PRIMARY KEY AUTOINCREMENT,
        rank INT NOT NULL,
        movement INT NOT NULL,
        points INT NOT NULL,
        competitions_played INT NOT NULL,
        competitor_id VARCHAR(50),
        FOREIGN KEY (competitor_id) REFERENCES Competitors(competitor_id)
    );
    """)

    # 2. Load CSV Data with NULL-value cleansing
    tables_map = {
        "Categories": "categories.csv",
        "Competitions": "competitions.csv",
        "Complexes": "complexes.csv",
        "Venues": "venues.csv",
        "Competitors": "competitors.csv",
        "Competitor_Rankings": "competitor_rankings.csv"
    }

    for table_name, csv_file in tables_map.items():
        df = pd.read_csv(csv_file)

        # Fill missing text values (NaN) with "N/A" to respect NOT NULL constraints
        string_cols = df.select_dtypes(include=['object']).columns
        df[string_cols] = df[string_cols].fillna("N/A")

        # Fill missing numeric values with 0
        num_cols = df.select_dtypes(include=['number']).columns
        df[num_cols] = df[num_cols].fillna(0)

        # Drop rank_id column if present so SQLite can autoincrement it
        if table_name == "Competitor_Rankings" and "rank_id" in df.columns:
            df = df.drop(columns=["rank_id"])

        df.to_sql(table_name, conn, if_exists="append", index=False)
        print(f"📥 Loaded {len(df)} rows into '{table_name}' table.")

    conn.commit()
    conn.close()
    print("✅ Database successfully created and populated!")

if __name__ == "__main__":
    build_database()

📥 Loaded 18 rows into 'Categories' table.
📥 Loaded 6619 rows into 'Competitions' table.
📥 Loaded 767 rows into 'Complexes' table.
📥 Loaded 4021 rows into 'Venues' table.
📥 Loaded 1000 rows into 'Competitors' table.
📥 Loaded 1000 rows into 'Competitor_Rankings' table.
✅ Database successfully created and populated!


In [10]:
categories = pd.read_csv('/home/dell/Documents/Game-Analytics-main/categories.csv')
categories.columns

Index(['category_id', 'category_name'], dtype='object')

In [11]:
categories.head(10)

,category_id,category_name
0,sr:category:181,Hopman Cup
1,sr:category:3,ATP
2,sr:category:72,Challenger
3,sr:category:6,WTA
4,sr:category:76,Davis Cup
5,sr:category:74,Billie Jean King Cup
6,sr:category:785,ITF Men
7,sr:category:213,ITF Women
8,sr:category:871,WTA 125K
9,sr:category:1012,IPTL


In [12]:
competitions = pd.read_csv('/home/dell/Documents/Game-Analytics-main/competitions.csv')
competitions.columns

Index(['competition_id', 'competition_name', 'parent_id', 'type', 'gender',
       'category_id'],
      dtype='object')

In [13]:
competitions.head(10)

,competition_id,competition_name,parent_id,type,gender,category_id
0,sr:competition:620,Hopman Cup,NaN,mixed,mixed,sr:category:181
1,sr:competition:660,World Team Cup,NaN,mixed,men,sr:category:3
2,sr:competition:990,ATP Challenger Tour Finals,sr:competition:6239,singles,men,sr:category:72
3,sr:competition:1207,Championship International Series,NaN,singles,women,sr:category:6
4,sr:competition:2100,Davis Cup,NaN,mixed,men,sr:category:76
5,sr:competition:2102,Billie Jean King Cup,NaN,mixed,women,sr:category:74
6,sr:competition:2555,Wimbledon Men Singles,sr:competition:2553,singles,men,sr:category:3
7,sr:competition:2557,Wimbledon Men Doubles,sr:competition:2553,doubles,men,sr:category:3
8,sr:competition:2559,Wimbledon Women Singles,sr:competition:2553,singles,women,sr:category:6
9,sr:competition:2561,Wimbledon Women Doubles,sr:competition:2553,doubles,women,sr:category:6


In [14]:
competitor_rankings = pd.read_csv('/home/dell/Documents/Game-Analytics-main/competitor_rankings.csv')
competitor_rankings.columns

Index(['rank', 'movement', 'points', 'competitions_played', 'competitor_id'], dtype='object')

In [15]:
competitor_rankings.head(10)

,rank,movement,points,competitions_played,competitor_id
0,1,0,9960,20,sr:competitor:14898
1,1,0,9960,20,sr:competitor:637970
2,3,0,7400,14,sr:competitor:16160
3,4,0,7310,15,sr:competitor:15568
4,5,0,7140,25,sr:competitor:53785
5,6,0,5500,27,sr:competitor:95801
6,7,0,5200,21,sr:competitor:15310
7,8,0,5050,21,sr:competitor:51836
8,9,0,5050,22,sr:competitor:49363
9,10,0,4920,19,sr:competitor:36593


In [16]:
competitors = pd.read_csv('/home/dell/Documents/Game-Analytics-main/competitors.csv')
competitors.columns

Index(['competitor_id', 'name', 'country', 'country_code', 'abbreviation'], dtype='object')

In [ ]:
competitors.head(10)

,competitor_id,name,country,country_code,abbreviation
0,sr:competitor:14898,"Heliovaara, Harri",Finland,FIN,HEL
1,sr:competitor:637970,"Patten, Henry",Great Britain,GBR,PAT
2,sr:competitor:16160,"Zeballos, Horacio",Argentina,ARG,ZEB
3,sr:competitor:15568,"Granollers, Marcel",Spain,ESP,GRA
4,sr:competitor:53785,"Skupski, Neal",Great Britain,GBR,SKU
5,sr:competitor:95801,"Vavassori, Andrea",Italy,ITA,VAV
6,sr:competitor:15310,"Bolelli, Simone",Italy,ITA,BOL
7,sr:competitor:51836,"Arevalo, Marcelo",El Salvador,SLV,ARE
8,sr:competitor:49363,"Pavic, Mate",Croatia,HRV,PAV
9,sr:competitor:36593,"Krawietz, Kevin",Germany,DEU,KRA


In [18]:
complexes = pd.read_csv('/home/dell/Documents/Game-Analytics-main/complexes.csv')
complexes.columns

Index(['complex_id', 'complex_name'], dtype='object')